# 01 — Data Exploration & Class Grouping

Loads the TDC DrugBank DDI dataset, explores it, and merges rare interaction types (<100 samples) into an `Other` bucket. Produces `df_grouped.pkl` for the next notebook plus the before/after class-distribution chart.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from tdc.multi_pred import DDI

data = DDI(name='DrugBank', path='../data')
df = data.get_data()
print(df.shape)
df.head()


## Explore the raw dataset

In [ ]:
print(df.shape)
print(df['Y'].value_counts())
print(df.isnull().sum())
df['Drug1'].sample(5).tolist()


## Group rare interaction types

Classes with fewer than 100 samples are merged into a single `Other` bucket — this keeps 53 of the original 86 classes distinct while ensuring every kept class has enough samples to survive a stratified train/val/test split. See `group_rare_labels` in `src/model.py` for the reusable logic.

In [ ]:
from src.model import group_rare_labels

df['Y_grouped'] = group_rare_labels(df, y_col='Y', threshold=100)
print(df['Y_grouped'].value_counts())
print(f"\nTotal classes: {df['Y_grouped'].nunique()}")


## Sanity checks

In [ ]:
# Row count integrity
assert df['Y_grouped'].value_counts().sum() == len(df)
assert df['Y_grouped'].isnull().sum() == 0
print("Row count check passed")

print(f"Total classes: {df['Y_grouped'].nunique()}")
print(f"Min class size: {df['Y_grouped'].value_counts().min()}")


In [ ]:
from tdc.utils import get_label_map

label_map = get_label_map(name='DrugBank', task='DDI', path='../data')

class_counts = df['Y'].value_counts()
rare_labels = class_counts[class_counts < 100].index
print("Labels merged into 'Other':")
for label_id in rare_labels:
    print(f"  {label_id}: {label_map[label_id]} ({class_counts[label_id]} samples)")


In [ ]:
# Confirm the largest class was left untouched
assert (df.loc[df['Y'] == 49, 'Y_grouped'] == 49).all()
print("Largest class unchanged: OK")


## Class distribution — before vs after grouping

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

before = df['Y'].value_counts().sort_values(ascending=False)
axes[0].bar(range(len(before)), before.values, color='steelblue')
axes[0].set_title(f'Original: {len(before)} classes')
axes[0].set_xlabel('Class rank')
axes[0].set_ylabel('Sample count')
axes[0].axhline(y=100, color='red', linestyle='--', linewidth=1, label='threshold=100')
axes[0].legend()

after = df['Y_grouped'].value_counts().sort_values(ascending=False)
axes[1].bar(range(len(after)), after.values, color='seagreen')
axes[1].set_title(f'Grouped: {len(after)} classes')
axes[1].set_xlabel('Class rank')
axes[1].set_ylabel('Sample count')

plt.tight_layout()
plt.savefig('../models/class_distribution_before_after.png', dpi=150)
plt.show()


## Save for the next notebook

In [ ]:
df.to_pickle('../data/df_grouped.pkl')
print("Saved ../data/df_grouped.pkl")
